# Técnicas de Análisis Multivariado (TAM)
### Pipeline Matricial: Medias, Varianzas, Covarianzas, Correlación y Kernels

Este cuaderno implementa el análisis multivariado mediante álgebra lineal pura (vectores de unos $\mathbf{1}$ y operador de centrado $):
* **Espacio de variables:**  	imes P$ (Primal: 84 	imes 784$)
* **Espacio de muestras:**  	imes N$ (Dual: 000 	imes 1000$)
* **Espacio RKHS:**  	imes N$ (Kernel RBF no lineal y su centrado  K H$)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

# 1. Carga de datos MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 2. Seleccionar 1000 muestras, aplanar (28*28 = 784) y normalizar [0, 1]
N_muestras = 1000
X = X_test[:N_muestras].reshape(-1, 28 * 28) / 255.0
y = y_test[:N_muestras]

# Ordenamos por etiqueta para ver bloques limpios por dígito en las matrices NxN
orden = np.argsort(y)
X = X[orden]
y = y[orden]

N, P = X.shape  # N = 1000 muestras, P = 784 características (píxeles)
print(f"Dimensiones de X: N = {N} muestras, P = {P} características (píxeles)")
print(f"Rango de valores de X: [{X.min():.1f}, {X.max():.1f}]")

### Paso 1: Vectores de Unos ($\mathbf{1}$) y Cálculo de Medias
Para realizar sumas y promedios matricialmente sin bucles:
* $\mathbf{1}_N \in \mathbb{R}^{N 	imes 1}$: Vector de unos para operar sobre las muestras.
* $\mathbf{1}_P \in \mathbb{R}^{P 	imes 1}$: Vector de unos para operar sobre los píxeles.
* **Media por variable (por píxel, sobre las $ muestras):** $oldsymbol{\mu}_n = rac{1}{N} \mathbf{1}_N^T X \in \mathbb{R}^{1 	imes P}$
* **Media por muestra (por imagen, sobre los $ píxeles):** $oldsymbol{\mu}_p = rac{1}{P} X \mathbf{1}_P \in \mathbb{R}^{N 	imes 1}$

In [ ]:
# Vectores de unos
On_N = np.ones((N, 1))
On_P = np.ones((P, 1))

# Medias matriciales
mu_n = (1 / N) * On_N.T @ X      # Media por variable (1, P) = (1, 784)
mu_p = (1 / P) * X @ On_P        # Media por muestra (N, 1) = (1000, 1)

print("Forma del vector de medias de variables (mu_n):", mu_n.shape)
print("Forma del vector de medias de muestras (mu_p):", mu_p.shape)

# Visualización: Dígito promedio reconstruido en 28x28
plt.figure(figsize=(4, 4))
plt.title("Dígito Promedio ($\mu_n$)")
plt.imshow(mu_n.reshape(28, 28), cmap="viridis")
plt.colorbar()
plt.show()

### Paso 2: Matriz de Centrado ($) y Matriz Centrada ($)
La matriz de centrado ortogonal proyecta los datos para que cada columna tenga media 0:
12398H_N = I_N - rac{1}{N} \mathbf{1}_N \mathbf{1}_N^T \in \mathbb{R}^{N 	imes N}12398
12398X_c = H_N X \in \mathbb{R}^{N 	imes P}12398
*Nota: $ es la matriz de desvíos respecto a la media. No es covarianza ni varianza todavía.*

In [ ]:
# Operador de centrado ortogonal
H_N = np.eye(N) - (1 / N) * (On_N @ On_N.T)  # (N, N)

# Matriz de datos centrada
Xc = H_N @ X  # (N, P)

print("Forma de X_c:", Xc.shape)
print("Comprobación de media cero (máximo desvío):", np.abs(Xc.mean(axis=0)).max())

# Visualización: Imagen original vs centrada
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title(f"Muestra Original (Dígito {y[0]})")
plt.imshow(X[0].reshape(28, 28), cmap="gray")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.title("Muestra Centrada ($)")
plt.imshow(Xc[0].reshape(28, 28), cmap="seismic")
plt.colorbar()
plt.tight_layout()
plt.show()

### Paso 3: Covarianzas en el Espacio Primal ( 	imes P$) y Dual ( 	imes N$)
* **Espacio de Características / Primal ( 	imes P$):** Covarianza entre píxeles normalizada por $:
12398\Sigma_{P 	imes P} = rac{1}{N} X_c^T X_c12398
* **Espacio de Muestras / Dual ( 	imes N$):** Covarianza / producto punto entre imágenes normalizada por $:
12398\Sigma_{N 	imes N} = rac{1}{P} X_c X_c^T12398

In [ ]:
# Covarianza de variables (PxP = 784x784)
Cov_P = (1 / N) * (Xc.T @ Xc)

# Covarianza de muestras (NxN = 1000x1000)
Cov_N = (1 / P) * (Xc @ Xc.T)

print("Dimensiones Cov_P (PxP):", Cov_P.shape)
print("Dimensiones Cov_N (NxN):", Cov_N.shape)

# Visualizaciones
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title("Covarianza de Características ( \times P$)")
plt.imshow(Cov_P, cmap="hot")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.title("Covarianza de Muestras ( \times N$)")
plt.imshow(Cov_N, cmap="hot")
plt.colorbar()
plt.tight_layout()
plt.show()

### Paso 4: Varianza y Matriz de Correlación ( 	imes P$)
* **Varianza:** Diagonal de la matriz de covarianza (varianza de cada píxel individualmente).
* **Desviación Estándar:**  = \sqrt{	ext{diag}(\Sigma_{P 	imes P})}$.
* **Correlación:** Matriz de covarianza normalizada para acotar los valores en 1$:
12398R_{P 	imes P} = rac{\Sigma_{P 	imes P}}{s s^T + \epsilon}12398

In [ ]:
# Varianzas (diagonal de Cov_P)
Var_P = np.diag(Cov_P).reshape(-1, 1)  # (784, 1)
std_P = np.sqrt(Var_P)                 # (784, 1)

# Matriz de correlación (con epsilon para evitar división por cero en píxeles constantes)
Corr_P = Cov_P / (std_P @ std_P.T + 1e-6)

# Visualizaciones
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title("Varianza por Píxel (28x28)")
plt.imshow(Var_P.reshape(28, 28), cmap="inferno")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.title("Matriz de Correlación ( \times P$)")
plt.imshow(Corr_P, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.tight_layout()
plt.show()

### Paso 5: Kernel RBF y Centrado en RKHS ( 	imes N$)
En el espacio de Hilbert $\mathcal{H}$ ($ dimensiones, donde  = \infty$), no calculamos la covarianza  	imes Q$.
En su lugar, usamos el **Kernel Trick** para calcular la matriz Gram en  	imes N$:
12398K_{ij} = \exp\left(-rac{\|x_i - x_j\|^2}{2\sigma^2}ight)12398
Y centramos en RKHS con el mismo operador $:
12398K_c = H_N K H_N12398

In [ ]:
def rbf_kernel(X1, X2=None, sigma=10.0):
    if X2 is None:
        X2 = X1
    # Distancia euclídea al cuadrado: ||x - y||^2 = ||x||^2 + ||y||^2 - 2<x, y>
    sq_dists = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * (X1 @ X2.T)
    return np.exp(-sq_dists / (2 * (sigma**2)))

# 1. Matriz de Kernel Gram (NxN = 1000x1000)
K = rbf_kernel(X, sigma=10.0)

# 2. Kernel Centrado usando la matriz H_N
K_c = H_N @ K @ H_N

# 3. Varianza y Correlación en RKHS
Var_K = np.diag(K_c).reshape(-1, 1)
Corr_K = K_c / (np.sqrt(Var_K @ Var_K.T) + 1e-6)

print("Dimensiones del Kernel K:", K.shape)

# Visualización comparativa
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Kernel RBF ($)")
plt.imshow(K, cmap="viridis")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.title("Kernel Centrado ($)")
plt.imshow(K_c, cmap="magma")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.title("Correlación Kernel ($)")
plt.imshow(Corr_K, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.tight_layout()
plt.show()